# NB1 – Retail-Daten kennenlernen, Teil A: Datenexploration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muehlburger/bid-ws2026/blob/main/notebooks/BID_NB1_TeilA.ipynb) Datensatz und Notebook: https://github.com/muehlburger/bid-ws2026
**Big Data (BID), WS 2026/27, Termin 1, Aufgabe 3, Paare, 25 Minuten**

**So arbeitet ihr**

1. **Zelle 1–3 einfach ausführen** (▶ links neben der Zelle) und lesen – sie laden den Datensatz und zeigen, wie er aufgebaut ist.
2. Danach **F1–F3** bearbeiten. In jeder Aufgabenzelle steht der Rechenweg fertig da; nur die mit `___` markierten Stellen ergänzt ihr (der Hinweis darüber sagt, womit).
3. Wer fertig ist: **Zusatzaufgabe** (Preisverteilung).

Regel im Raum: Wenn bei euch etwas nicht lädt, arbeitet beim Nachbarpaar mit – Setup-Probleme lösen wir nicht im Plenum.

**Der Datensatz** ist vollständig synthetisch (keine Kundendaten), aber im Schema eines typischen Warenwirtschafts-Exports: 12 Filialen (Filiale 1 = Onlineshop), 800 Artikelvarianten, 1.7.2024–30.6.2026, rund 400.000 Zeilen. **Eine Zeile = Tagesumsatz einer Variante in einer Filiale.**

In [ ]:
# Cell 1 – Load the dataset (just run)
import os, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/muehlburger/bid-ws2026/main/data/clean/"  # public repo with the synthetic dataset (base URL)
FILES = ["variantenumsatz.parquet", "Filiale.csv"]

def fetch(name):
    """Look for the file next to the notebook first, then under DATA_URL, otherwise open the upload dialog (Colab only)."""
    if os.path.exists(name):
        return name
    if not DATA_URL.startswith("https://<"):
        urllib.request.urlretrieve(DATA_URL + name, name)
        return name
    try:
        from google.colab import files  # only available in Colab
        print(f"Please upload {name} from Moodle:")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"{name} not found – put the file from Moodle next to the notebook.")

___ = None  # placeholder: in the tasks below, replace ___ with your expression

def filled(value, label):
    """Return True if a task placeholder was replaced; otherwise print a hint instead of failing."""
    if value is None:
        print(f"{label}: not filled in yet – replace ___ above")
        return False
    return True

df = pd.read_parquet(fetch("variantenumsatz.parquet"))
print("Rows and columns:", df.shape)
df.head(3)

In [ ]:
# Cell 2 – Period and size (just run)
df["date"] = pd.to_datetime(df["date_bestand_ums"])
print("Period:", df["date"].min().date(), "to", df["date"].max().date())
print("Stores:", df["filialnummer3"].nunique(), "| Variants:", df["variantennummer8"].nunique(),
      "| Days with sales:", df["date"].nunique())
df.dtypes

**Eine Zeile erklärt** (die wichtigsten Spalten):

| Spalte | Bedeutung |
|---|---|
| `date_bestand_ums` | Buchungstag |
| `variantennummer8` | Artikelvariante (Artikel × Größe × Farbe) |
| `filialnummer3` | Filiale (1 = Onlineshop) |
| `verkaufspreis_ifw` | regulärer (Listen-)Verkaufspreis je Stück in € |
| `umsatz_menge_mk9` | verkaufte Stück – **negativ = Retoure** |
| `umsatz_wert_ifw` | Umsatz in € = Menge × regulärer Preis − Reduktionen (`redukt_*`); bezahlter Stückpreis = Umsatz ÷ Menge |
| `kz_aktion` | `A` = Aktions-/Sale-Verkauf, `R` = regulär |
| `kz_saison6` | Saison: `202531` = Frühjahr/Sommer 2025, `202532` = Herbst/Winter 2025 |

In [ ]:
# Cell 3 – Master data: what is behind a store number? (just run)
stores = pd.read_csv(fetch("Filiale.csv"), sep=";", decimal=",")
df = df.merge(stores[["filialnummer3", "filial_bezeichnung", "hw_online"]], on="filialnummer3", how="left")
df.groupby(["filialnummer3", "filial_bezeichnung"])["umsatz_menge_mk9"].sum().sort_values(ascending=False)

## F1 – Skalenniveau und Diagrammwahl

Bestimme das **Skalenniveau** (nominal / ordinal / metrisch) von `filialnummer3`, `verkaufspreis_ifw`, `kz_aktion` und `date_bestand_ums`.
Welches **Diagramm** eignet sich für (a) Absatzmenge je Wochentag, (b) Verteilung des Verkaufspreises, (c) Anteil Aktions- vs. Normalverkäufe?

Führt die Zelle aus, schaut euch Typen und Beispielwerte an und schreibt die Antwort in die Textzelle darunter (Doppelklick zum Bearbeiten).

In [ ]:
# F1 – Inspect data types and sample values (careful: the data type is not the level of measurement!)
columns = ["filialnummer3", "verkaufspreis_ifw", "kz_aktion", "date_bestand_ums"]
print(df[columns].dtypes)
df[columns].sample(5, random_state=1)

**Eure Antwort F1**

- `filialnummer3`: …
- `verkaufspreis_ifw`: …
- `kz_aktion`: …
- `date_bestand_ums`: …
- Diagramme: (a) …; (b) …; (c) …

## F2 – Median vs. Mittelwert

Berechne **Median und Mittelwert der Tagesmenge je Variante und Filiale** (nur Verkäufe, Menge > 0).
Warum unterscheiden sie sich – und welche Kenngröße würdest du für „typischer Tagesabsatz“ berichten?

*Hinweis:* Die Tagesmenge je Variante und Filiale ist die Summe je `variantennummer8`, `filialnummer3` und `date`. Für die beiden Kenngrößen gibt es die Methoden `.median()` und `.mean()`.

In [ ]:
# F2 – Daily quantity per variant × store
sales = df[df["umsatz_menge_mk9"] > 0]                                       # sales only (returns are negative)
daily_qty = sales.groupby(["variantennummer8", "filialnummer3", "date"])["umsatz_menge_mk9"].sum()

median = ___   # ← fill in: daily_qty.<method>()
mean = ___     # ← fill in: daily_qty.<method>()

if filled(median, "Median"): print("Median: ", median)
if filled(mean, "Mean"):     print("Mean:   ", round(mean, 2))

daily_qty.clip(upper=10).plot.hist(bins=range(1, 12), rwidth=0.9, title="Daily quantity per variant × store (units, ≥10 pooled)")
plt.xlabel("Units per day"); plt.ylabel("Number of sales days"); plt.show()

**Eure Antwort F2:** Median = …; Mittelwert = …; Grund für den Unterschied: …; Für „typischer Tagesabsatz“ berichten wir: …

## F3 – Wochentagsprofil

Erstelle das **Wochentagsprofil der Gesamtmenge**. Welcher Tag ist der stärkste, welcher zeigt null – und was bedeutet das für eine Absatzprognose?

*Hinweis:* `dt.dayofweek` liefert 0 = Montag … 6 = Sonntag. Schaut euch zuerst das Ergebnis der Gruppierung an: **Welcher Wochentag fehlt – und warum?** Mit `.reindex(range(7), fill_value=0)` macht ihr alle sieben Tage sichtbar – überlegt dabei, woher ihr wisst, dass `0` die richtige Füllung ist.

In [ ]:
# F3 – Weekday profile of the total quantity
df["weekday"] = df["date"].dt.dayofweek                          # 0 = Monday … 6 = Sunday
profile = df.groupby("weekday")["umsatz_menge_mk9"].sum()
print(profile)                                                   # ← Which day is missing here? Why?

profile_all = ___   # ← fill in: profile.<method>(range(7), fill_value=0) – make all seven weekdays visible

if filled(profile_all, "Weekday profile"):
    profile_all.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    profile_all.plot.bar(title="Sales quantity per weekday (units)", rot=0); plt.ylabel("Units"); plt.show()

**Eure Antwort F3:** stärkster Tag = …; zweitstärkster = …; null am … weil …; Für die Prognose bedeutet das: …

## Zusatz (für schnelle Paare) – Verteilung des Verkaufspreises

Zeichne die Verteilung des regulären Verkaufspreises und benenne Preislagen. Welche Kenngröße beschreibt den „typischen Preis“ besser – Median oder Mittelwert?

In [ ]:
# Extra – Distribution of the sales price (runs without changes)
prices = df["verkaufspreis_ifw"]
print("Median:", prices.median(), "| Mean:", round(prices.mean(), 2), "| Minimum:", prices.min(), "| Maximum:", prices.max())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
prices.plot.hist(bins=40, ax=ax[0], title="Histogram of the sales price (€)")
prices.plot.box(ax=ax[1], title="Box plot of the sales price (€)")
plt.show()
print("The ten most frequent prices:")
print(prices.round(2).value_counts().head(10))